In [20]:
import shutil
from pathlib import Path

# Remove the old processed data folder/file if it exists
processed_dir = Path("../data/processed")
if processed_dir.exists():
    shutil.rmtree(processed_dir)
    print("Deleted old processed data.")

Deleted old processed data.


In [21]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

BASE = "https://api.ukhsa-dashboard.data.gov.uk"

def get_amr_metric(topic, metric, geography_type="UKHSA Region", page_size=365):
    geo_url = f"{BASE}/themes/infectious_disease/sub_themes/antimicrobial_resistance/topics/{topic}/geography_types/{geography_type}/geographies?format=json"
    geos = [g["name"] for g in requests.get(geo_url).json()]
    
    frames = []
    for geo in geos:
        url = (f"{BASE}/themes/infectious_disease/sub_themes/antimicrobial_resistance"
               f"/topics/{topic}/geography_types/{geography_type}/geographies/{geo}/metrics/{metric}"
               f"?format=json&page_size={page_size}")
        results = []
        while url:
            r = requests.get(url)
            r.raise_for_status()
            data = r.json()
            results.extend(data.get("results", []))
            url = data.get("next")
        if results:
            df = pd.DataFrame(results)
            df = df.rename(columns={"date":"date", "geography":"region", "metric_value":"resistance",
                                    "stratum":"resistance_type", "topic":"bacteria"})
            df = df[["date", "region", "resistance", "bacteria", "resistance_type"]]
            df["bacteria"] = df["bacteria"].str.replace("-", "_")
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

# ========== 1. Download fresh data for all bacteria ==========
CONFIG = [
    {"topic": "E-coli", "metric": "e-coli_testing_bacteraemiaPercentResistantRollingMonth", "label": "E_coli"},
    {"topic": "E-faecalis", "metric": "e-faecalis_testing_bacteraemiaPercentResistantRollingMonth", "label": "E_faecalis"},
    {"topic": "E-faecium", "metric": "e-faecium_testing_bacteraemiaPercentResistantRollingMonth", "label": "E_faecium"},
    {"topic": "K-pneumoniae", "metric": "k-pneumoniae_testing_bacteraemiaPercentResistantRollingMonth", "label": "K_pneumoniae"},
    {"topic": "P-aeruginosa", "metric": "p-aeruginosa_testing_bacteraemiaPercentResistantRollingMonth", "label": "P_aeruginosa"},
    {"topic": "S-pneumoniae", "metric": "s-pneumoniae_testing_bacteraemiaPercentResistantRollingMonth", "label": "S_pneumoniae"},
]

frames = []
for cfg in CONFIG:
    print(f"Downloading {cfg['label']} ...")
    df = get_amr_metric(cfg["topic"], cfg["metric"])
    df["bacteria"] = cfg["label"]
    print(f"  → {len(df)} rows")
    frames.append(df)

amr = pd.concat(frames, ignore_index=True)
amr["date"] = pd.to_datetime(amr["date"])

# ========== 2. Reindex to guarantee every bacteria has every region & date ==========
all_bacteria = amr["bacteria"].unique()
all_resistance = amr["resistance_type"].unique()
all_regions = amr["region"].unique()
all_dates = amr["date"].unique()

complete_index = pd.MultiIndex.from_product(
    [all_bacteria, all_resistance, all_regions, all_dates], 
    names=["bacteria", "resistance_type", "region", "date"]
)

amr = amr.set_index(["bacteria", "resistance_type", "region", "date"]) \
         .reindex(complete_index) \
         .reset_index()

amr = amr.sort_values(["bacteria", "resistance_type", "region", "date"])

# Save clean master file
csv_path = Path("../data/processed/amr_all_bacteria.csv")
csv_path.parent.mkdir(parents=True, exist_ok=True)
amr.to_csv(csv_path, index=False)

print("\nDataset successfully rebuilt from scratch!")
print(amr["bacteria"].value_counts())

# ========== 3. Interactive Dashboard Widgets & Plotting ==========
bacteria_dropdown = widgets.Dropdown(
    options=sorted(amr["bacteria"].unique()),
    description="Bacteria:",
    layout=widgets.Layout(width="220px")
)

resistance_dropdown = widgets.Dropdown(
    options=[],
    description="Resistance:",
    layout=widgets.Layout(width="280px")
)

region_dropdown = widgets.Dropdown(
    options=["All Regions"] + sorted(amr["region"].dropna().unique().tolist()),
    value="All Regions",
    description="Region:",
    layout=widgets.Layout(width="200px")
)

output = widgets.Output()

def update_plot(change=None):
    with output:
        clear_output(wait=True)
        bact = bacteria_dropdown.value
        rest = resistance_dropdown.value
        
        if not bact or not rest:
            print("Please make a selection.")
            return

        df = amr[(amr["bacteria"] == bact) & (amr["resistance_type"] == rest)]
        
        if region_dropdown.value != "All Regions":
            df = df[df["region"] == region_dropdown.value]
        
        df = df.dropna(subset=["resistance"])
        
        if df.empty:
            print("No data available for this selection.")
            return
        
        plt.figure(figsize=(13, 6))
        
        if region_dropdown.value == "All Regions":
            for region in sorted(df["region"].unique()):
                subset = df[df["region"] == region].sort_values("date")
                if not subset.empty:
                    plt.plot(subset["date"], subset["resistance"], label=region, linewidth=1.8)
            plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Region", fontsize=8)
        else:
            plt.plot(df.sort_values("date")["date"], df.sort_values("date")["resistance"], 
                     color="steelblue", linewidth=2.5)
        
        plt.title(f"{bact.replace('_', ' ')} – {rest}", fontsize=14)
        plt.ylabel("% Resistant")
        plt.grid(True, linestyle="--", alpha=0.4)
        plt.tight_layout()
        plt.show()

def update_resistance(change):
    bact = change["new"]
    available = sorted(amr[(amr["bacteria"] == bact) & (amr["resistance"].notna())]["resistance_type"].unique())
    resistance_dropdown.options = available
    if available:
        resistance_dropdown.value = available[0]
    update_plot()

# Attach observers
bacteria_dropdown.observe(update_resistance, names="value")
resistance_dropdown.observe(update_plot, names="value")
region_dropdown.observe(update_plot, names="value")

# Initialize dropdown values
if len(bacteria_dropdown.options) > 0:
    bacteria_dropdown.value = bacteria_dropdown.options[0]

display(widgets.HBox([bacteria_dropdown, resistance_dropdown, region_dropdown]))
display(output)

  → 4788 rows
  → 684 rows
  → 684 rows
  → 4787 rows
  → 3951 rows
  → 1362 rows

Dataset successfully rebuilt from scratch!
bacteria
E_coli          7524
E_faecalis      7524
E_faecium       7524
K_pneumoniae    7524
P_aeruginosa    7524
S_pneumoniae    7524
Name: count, dtype: int64


Output()